In [ ]:
import torch
import pandas as pd
import torch.nn as nn
from transformers import BertForSequenceClassification, BertTokenizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
MODEL_PATH = "AungMoonLord/bert-log-anomaly-detection"

model = BertForSequenceClassification.from_pretrained(MODEL_PATH)
tokenizer = BertTokenizer.from_pretrained(MODEL_PATH)

df = pd.read_csv("/content/Dataset (claude)_2 - Sheet1.csv")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [ ]:
# Label Mapping
label_map = {
    "Anomaly": 0,
    "Normal": 1
}

In [ ]:
y_true = []
y_pred = []
total_loss = 0.0

criterion = nn.CrossEntropyLoss()

for index, row in df.iterrows():
    text = row['query log']
    true_label_str = row['status']
    true_label = label_map[true_label_str]

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    labels = torch.tensor([true_label]).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

        pred = torch.argmax(logits, dim=1).item()
        prob = torch.softmax(logits, dim=-1).tolist()[0]

        loss = criterion(logits, labels)

    y_true.append(true_label)
    y_pred.append(pred)
    total_loss += loss.item()

    prediction_result = "Normal" if pred == 1 else "Anomaly"
    correction = "True" if prediction_result == true_label_str else "False"

    print(
        f"prediction = {prediction_result} | "
        f"true_status = {true_label_str} | "
        f"correction = {correction} | "
        f"confidence = {prob}"
    )

# Metrics
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
val_loss = total_loss / len(df)

prediction = Anomaly | true_status = Anomaly | correction = True | confidence = [0.8568318486213684, 0.1431681513786316]
prediction = Anomaly | true_status = Anomaly | correction = True | confidence = [0.6062373518943787, 0.39376264810562134]
prediction = Anomaly | true_status = Anomaly | correction = True | confidence = [0.957196831703186, 0.04280320554971695]
prediction = Anomaly | true_status = Anomaly | correction = True | confidence = [0.953289270401001, 0.046710751950740814]
prediction = Normal | true_status = Anomaly | correction = False | confidence = [0.17400753498077393, 0.8259925246238708]
prediction = Anomaly | true_status = Anomaly | correction = True | confidence = [0.5630027651786804, 0.43699729442596436]
prediction = Anomaly | true_status = Anomaly | correction = True | confidence = [0.9653347134590149, 0.03466532751917839]
prediction = Anomaly | true_status = Anomaly | correction = True | confidence = [0.9831695556640625, 0.016830485314130783]
prediction = Anomaly | tr

In [ ]:
# Result
print("\n===== Evaluation Result =====")
print(f"Accuracy        : {accuracy:.4f}")
print(f"Precision       : {precision:.4f}")
print(f"Recall          : {recall:.4f}")
print(f"F1-score        : {f1:.4f}")
print(f"Validation Loss : {val_loss:.4f}")


===== Evaluation Result =====
Accuracy        : 0.6950
Precision       : 0.6639
Recall          : 0.7900
F1-score        : 0.7215
Validation Loss : 0.6251


In [ ]:
# วัด Accuracy
total_predictions = len(y_true)
correct_predictions = sum(1 for true, pred in zip(y_true, y_pred) if true == pred)
accuracy = (correct_predictions / total_predictions) * 100
print(f"test_set จำนวน {total_predictions}")
print(f"ทำนายถูกจำนวน {correct_predictions}")
print(f"Accuracy = {accuracy:.2f}%")

test_set จำนวน 200
ทำนายถูกจำนวน 139
Accuracy = 69.50%
